Creating and running a graph

In [1]:
import tensorflow as tf

In [2]:
x = tf.Variable(3,name = "x")
y = tf.Variable(5,name = "y")

In [3]:
f=x*x*y+y+2

In [4]:
sess= tf.Session()
sess.run(x.initializer)
sess.run(y.initializer)
result=sess.run(f)

In [5]:
result

52

In [6]:
sess.close()

In [7]:
x = tf.Variable(3,name = "x")
y = tf.Variable(5,name = "y")

In [8]:
init = tf.global_variables_initializer()

In [9]:
with tf.Session() as sess:
    init.run()
    result = f.eval()

In [10]:
result

52

In [11]:
sess= tf.InteractiveSession()

In [12]:
init.run()

In [13]:
result = f.eval()

In [14]:
result

52

In [15]:
sess.close()

In [16]:
import numpy as np

In [17]:
# to make this notebook's output stable across runs
def reset_graph(seed=42):
    tf.reset_default_graph()
    tf.set_random_seed(seed)
    np.random.seed(seed)

In [18]:
reset_graph()

In [19]:
x1 = tf.Variable(1)

In [20]:
x1.graph is tf.get_default_graph()

True

In [21]:
graph = tf.Graph()

In [22]:
with graph.as_default():
    x2 = tf.Variable(2)

In [23]:
x2.graph is graph

True

In [24]:
x2.graph is tf.get_default_graph()

False

In [25]:
w = tf.constant(3)
x = w + 2
y = x + 5
z = x * 3

with tf.Session() as sess:
    print(y.eval())  # 10
    print(z.eval())  # 15

10
15


In [26]:
with tf.Session() as sess:
    y_val, z_val = sess.run([y, z])
    print(y_val)  # 10
    print(z_val)  # 15

10
15


Linear Regression with Tensorflow

In [27]:
import numpy as np
from sklearn.datasets import fetch_california_housing

In [28]:
housing = fetch_california_housing()

In [29]:
m,n = housing.data.shape

In [30]:
m

20640

In [31]:
n

8

In [32]:
housing_data_plus_bias = np.c_[np.ones((m,1)), housing.data]

In [33]:
X = tf.constant(housing_data_plus_bias, dtype=tf.float32, name="X")
y = tf.constant(housing.target.reshape(-1, 1), dtype=tf.float32, name="y")

In [34]:
XT = tf.transpose(X)

In [35]:
theta = tf.matmul(tf.matmul(tf.matrix_inverse(tf.matmul(XT,X)),XT),y)

In [36]:
with tf.Session() as sess:
    theta_value = theta.eval()

In [37]:
theta_value

array([[ -3.74651413e+01],
       [  4.35734153e-01],
       [  9.33829229e-03],
       [ -1.06622010e-01],
       [  6.44106984e-01],
       [ -4.25131839e-06],
       [ -3.77322501e-03],
       [ -4.26648885e-01],
       [ -4.40514028e-01]], dtype=float32)

Compare with numpy

In [38]:
import numpy as np

In [39]:
X = housing_data_plus_bias
y = housing.target.reshape(-1,1)

In [40]:
theta_numpy = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)

In [41]:
theta_numpy

array([[ -3.69419202e+01],
       [  4.36693293e-01],
       [  9.43577803e-03],
       [ -1.07322041e-01],
       [  6.45065694e-01],
       [ -3.97638942e-06],
       [ -3.78654265e-03],
       [ -4.21314378e-01],
       [ -4.34513755e-01]])

Compare with scikit

In [42]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()

In [43]:
lin_reg.fit(housing.data, housing.target.reshape(-1, 1))

LinearRegression(copy_X=True, fit_intercept=True, n_jobs=1, normalize=False)

In [44]:
np.r_[lin_reg.intercept_.reshape(-1, 1), lin_reg.coef_.T]

array([[ -3.69419202e+01],
       [  4.36693293e-01],
       [  9.43577803e-03],
       [ -1.07322041e-01],
       [  6.45065694e-01],
       [ -3.97638942e-06],
       [ -3.78654265e-03],
       [ -4.21314378e-01],
       [ -4.34513755e-01]])

Implementing Gradient Descent

In [66]:
#but before using Gradient Descent, we must normalize the input feature vector
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [67]:
scaled_housing_data = scaler.fit_transform(housing.data)
scaled_housing_data_plus_bias = np.c_[np.ones((m, 1)), scaled_housing_data]

test some value

In [68]:
scaled_housing_data_plus_bias.mean(axis=0)#mean of each column

array([  1.00000000e+00,   6.60969987e-17,   5.50808322e-18,
         6.60969987e-17,  -1.06030602e-16,  -1.10161664e-17,
         3.44255201e-18,  -1.07958431e-15,  -8.52651283e-15])

In [69]:
scaled_housing_data_plus_bias.mean(axis=1)#mean of each line

array([ 0.38915536,  0.36424355,  0.5116157 , ..., -0.06612179,
       -0.06360587,  0.01359031])

In [70]:
scaled_housing_data_plus_bias.mean()

0.11111111111111005

In [71]:
scaled_housing_data_plus_bias.shape

(20640, 9)

Manually computing the graph

In [72]:
reset_graph()

this use gradient descent to find the best theta for Linear Regression

In [73]:
#number of epoch and learning rate
n_epochs = 1000
learning_rate = 0.01

X = tf.constant(scaled_housing_data_plus_bias, dtype=tf.float32, name ="X")
y = tf.constant(housing.target.reshape(-1,1), dtype = tf.float32, name = "y")

In [74]:
#Initialize theta, shape [9,1] from -1.0 to 1.0
theta = tf.Variable(tf.random_uniform([n+1,1], -1.0, 1.0), name = "theta")

In [75]:
y_pred = tf.matmul(X,theta, name = "Predictions")
error = y_pred -y

In [76]:
# minimum square error
mse = tf.reduce_mean(tf.square(error), name = "mse")

In [77]:
#gradient
gradients =  2/m*tf.matmul(tf.transpose(X), error)

In [78]:
training_op = tf.assign(theta, theta-learning_rate * gradients)

In [79]:
init = tf.global_variables_initializer()

In [80]:
with tf.Session() as sess:
    sess.run(init)
    
    for epoch in range(n_epochs):
        if epoch%100 ==0:
            print("Epoch", epoch, "MSE = ", mse.eval())
        sess.run(training_op)
    
    best_theta = theta.eval()

Epoch 0 MSE =  12.408
Epoch 100 MSE =  0.755197
Epoch 200 MSE =  0.542087
Epoch 300 MSE =  0.53317
Epoch 400 MSE =  0.530538
Epoch 500 MSE =  0.528797
Epoch 600 MSE =  0.527548
Epoch 700 MSE =  0.52665
Epoch 800 MSE =  0.526001
Epoch 900 MSE =  0.525533


In [81]:
best_theta

array([[  2.06855249e+00],
       [  8.10635984e-01],
       [  1.26857743e-01],
       [ -2.07840845e-01],
       [  2.48398483e-01],
       [ -1.30839343e-03],
       [ -3.96070518e-02],
       [ -8.58612776e-01],
       [ -8.26002836e-01]], dtype=float32)

Using autodiff

In [82]:
reset_graph()

repeat all above process except gradient calculation

In [84]:
reset_graph()

n_epochs = 1000
learning_rate = 0.01

X = tf.constant(scaled_housing_data_plus_bias, dtype=tf.float32, name="X")
y = tf.constant(housing.target.reshape(-1, 1), dtype=tf.float32, name="y")
theta = tf.Variable(tf.random_uniform([n + 1, 1], -1.0, 1.0, seed=42), name="theta")
y_pred = tf.matmul(X, theta, name="predictions")
error = y_pred - y
mse = tf.reduce_mean(tf.square(error), name="mse")

In [86]:
gradients = tf.gradients(mse, [theta])[0]

In [87]:
training_op = tf.assign(theta, theta- learning_rate*gradients)

In [88]:
init = tf.global_variables_initializer()

In [89]:
with tf.Session() as sess:
    sess.run(init)
    
    for epoch in range(n_epochs):
        if epoch % 100 ==0:
            print("Epoch", epoch, "MSE = ", mse.eval())
        sess.run(training_op)
    
    best_theta = theta.eval()

Epoch 0 MSE =  9.16154
Epoch 100 MSE =  0.714501
Epoch 200 MSE =  0.566705
Epoch 300 MSE =  0.555572
Epoch 400 MSE =  0.548812
Epoch 500 MSE =  0.543636
Epoch 600 MSE =  0.539629
Epoch 700 MSE =  0.536509
Epoch 800 MSE =  0.534068
Epoch 900 MSE =  0.532147


In [90]:
best_theta

array([[ 2.06855249],
       [ 0.88740271],
       [ 0.14401658],
       [-0.34770882],
       [ 0.36178368],
       [ 0.00393811],
       [-0.04269556],
       [-0.66145277],
       [-0.6375277 ]], dtype=float32)